# Volatility Dynamics 06b — latent-state HMM, repaired, against a volatility benchmark

Claude review (**VD 010**). Module 06's headline (state 3 = 3.7× q99 hazard) rested on 5 held-out events at 3 timestamps (2 of them aftershocks), K sat at the grid edge, sequences broke every ~50 min, and state occupancy drifted with the IV level. There was also no benchmark: a high-activity state predicting q99 is just volatility clustering unless it beats a trivial volatility rule.

06b (`volatility_hmm_states_b.py`): log emissions; IV as causal deviation from its trailing 24h median (ATM level + 25d skew); asset × UTC-hour training-only normalization; missing emissions marginalized (sequences break only on gaps); K chosen by validation log-likelihood; filtered probabilities only on the held-out period; like-for-like simulated duration check.

**Pre-registered decision (primary $h=60$ min):** the HMM carries q99 information beyond volatility iff
$\Delta=\text{logloss}(M3)-\text{logloss}(M1)<0$ **and** its 95% day-block bootstrap upper bound $<0$, with ≥ 20 unique held-out onset timestamps (else `INSUFFICIENT_DATA`).
$M1$ = asset base rates + $\log RV_{15m}, \log RV_{1h}, \log\text{range}_t, \log\overline{\text{range}}_{15m}$; $M3$ = $M1$ + filtered state probabilities.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_hmm_states_b as hb
START = os.getenv('VD6B_START', '').strip() or None
END = os.getenv('VD6B_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD6B_ASSETS', '').split(',') if x.strip()]
TRAIN_FRACTION = float(os.getenv('VD6B_TRAIN_FRACTION', str(hb.DEFAULT_TRAIN_FRACTION)))
K_VALUES = tuple(int(x) for x in os.getenv('VD6B_K', ','.join(map(str, hb.DEFAULT_K))).split(',') if x.strip())
N_INIT = int(os.getenv('VD6B_N_INIT', str(hb.DEFAULT_N_INIT)))
MAX_ITER = int(os.getenv('VD6B_MAX_ITER', str(hb.DEFAULT_MAX_ITER)))
N_BOOT = int(os.getenv('VD6B_N_BOOT', '2000'))
N_SIMS = int(os.getenv('VD6B_N_SIMS', '30'))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '06b'
OUT.mkdir(parents=True, exist_ok=True)
print('06b', hb.VERSION, hb.selftest())


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

schema = set(schema_columns()['column_name'].astype(str))
missing = set(hb.REQUIRED_COLUMNS) - schema
if missing:
    raise RuntimeError(f'missing columns: {missing}')
df = load_frame(list(hb.REQUIRED_COLUMNS), START, END, ASSETS)
grid = vb.build_grid(df)


In [ ]:
E = hb.build_emissions(grid, include_l2=False)
split = hb.chronological_split(grid, TRAIN_FRACTION)
train_mask = grid.time_ns < split.test_start_ns
params = hb.fit_hour_normalizer(grid, E, train_mask)
Z = hb.apply_normalizer(grid, E, params)
display(pd.DataFrame({'emission': E.columns, 'train_coverage': np.isfinite(E.to_numpy())[train_mask].mean(axis=0).round(3),
                      'test_coverage': np.isfinite(E.to_numpy())[~train_mask].mean(axis=0).round(3)}))
fit_idx = hb.sequences(grid, grid.time_ns < split.valid_start_ns)
val_idx = hb.sequences(grid, (grid.time_ns >= split.valid_start_ns) & train_mask)
tr_idx = hb.sequences(grid, train_mask)
te_idx = hb.sequences(grid, ~train_mask)
seq_meta = pd.DataFrame([{'split': s, 'n_sequences': len(ix), 'rows': int(sum(map(len, ix))),
                          'median_len': float(np.median([len(i) for i in ix])) if ix else 0}
                         for s, ix in [('fit', fit_idx), ('valid', val_idx), ('train', tr_idx), ('test', te_idx)]])
display(seq_meta)
print('test starts', pd.Timestamp(split.test_start_ns, tz='UTC'), '| validation starts', pd.Timestamp(split.valid_start_ns, tz='UTC'))


In [ ]:
t0 = time.time()
comparison, model, k_sel = hb.select_and_fit([Z[i] for i in fit_idx], [Z[i] for i in val_idx], [Z[i] for i in tr_idx],
                                             K_VALUES, n_init=N_INIT, max_iter=MAX_ITER, progress=True)
print(f'fit {time.time()-t0:.0f}s | selected K = {k_sel}')
display(comparison.round(4))
emis = pd.DataFrame(model.means, columns=E.columns).rename_axis('state').round(2)
display(emis)
display(pd.DataFrame(model.A).round(3))


In [ ]:
ft_train = hb.filtered_table(grid, model, tr_idx, Z, 'train')
ft_test = hb.filtered_table(grid, model, te_idx, Z, 'test')
thr = vb.q99_thresholds(grid, row_mask=train_mask)          # training-only thresholds
tail = vb.tail_flags(grid, thr)
onset = vb.onset_flags(grid, tail)
labels, first_event = hb.future_onset_labels(grid, onset)
bench = hb.benchmark_features(grid)
def assemble(ft):
    t = ft.copy()
    t['underlying'] = [grid.assets[c] for c in grid.asset_code[t.row]]
    t['day'] = grid.day[t.row]
    t = pd.concat([t.reset_index(drop=True), labels.iloc[t.row].reset_index(drop=True), bench.iloc[t.row].reset_index(drop=True)], axis=1)
    return t[np.isfinite(t[list(hb.BENCHMARK_FEATURES)]).all(axis=1)].reset_index(drop=True)
tr_tab, te_tab = assemble(ft_train), assemble(ft_test)
test_onsets = np.flatnonzero(onset & ~train_mask)
n_unique_test = len(np.unique(grid.time_ns[test_onsets]))
print('train rows', len(tr_tab), '| test rows', len(te_tab), '| test onsets', len(test_onsets), '| unique test onset timestamps', n_unique_test)
display(ft_test.groupby(grid.day[ft_test.row]).hard_state.value_counts(normalize=True).unstack().round(2))


In [ ]:
state_cols = [f'p_state_{k}' for k in range(model.K)]
scores, boot = hb.benchmark_comparison(tr_tab, te_tab, state_cols, n_boot=N_BOOT)
decision = hb.hmm_decision(boot, n_unique_test)
display(scores.round(5)); display(boot.round(5))
print(json.dumps(decision, indent=2))
hazard = hb.state_hazard_with_events(te_tab, model.K, first_event, grid)
display(hazard.round(4))


In [ ]:
t0 = time.time()
dur_summary, dur_hazard = hb.duration_check(model, [Z[i] for i in te_idx], n_sims=N_SIMS)
print(f'duration simulation {time.time()-t0:.0f}s')
display(dur_summary.round(3))
align = hb.event_alignment(grid, ft_test, test_onsets, model.K)
if len(align):
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for c in state_cols:
        ax.plot(align.rel_min, align[c], label=c)
    ax.axvline(0, linewidth=1); ax.axvline(-95, linestyle='--', linewidth=1)
    ax.set_xlabel('minutes relative to held-out q99 onset'); ax.legend(ncol=4); plt.show()


In [ ]:
params.to_csv(OUT / '06b_hour_normalizer.csv', index=False)
seq_meta.to_csv(OUT / '06b_sequence_metadata.csv', index=False)
comparison.to_csv(OUT / '06b_k_selection.csv', index=False)
pd.DataFrame(model.means, columns=E.columns).rename_axis('state').reset_index().to_csv(OUT / '06b_emission_means.csv', index=False)
pd.DataFrame(np.sqrt(model.variances), columns=E.columns).rename_axis('state').reset_index().to_csv(OUT / '06b_emission_sds.csv', index=False)
pd.DataFrame(model.A).rename_axis('from_state').reset_index().to_csv(OUT / '06b_transitions.csv', index=False)
ft_test.to_csv(OUT / '06b_filtered_test.csv', index=False)
pd.DataFrame([{'underlying': k, 'q99_threshold_train': v} for k, v in thr.items()]).to_csv(OUT / '06b_q99_train_thresholds.csv', index=False)
scores.to_csv(OUT / '06b_benchmark_scores.csv', index=False)
boot.to_csv(OUT / '06b_benchmark_bootstrap.csv', index=False)
hazard.to_csv(OUT / '06b_state_q99_hazard.csv', index=False)
dur_summary.to_csv(OUT / '06b_duration_summary.csv', index=False)
dur_hazard.to_csv(OUT / '06b_duration_hazard.csv', index=False)
align.to_csv(OUT / '06b_q99_alignment.csv', index=False)
(OUT / '06b_decision.json').write_text(json.dumps(decision, indent=2))
meta = {
    'version': hb.VERSION, 'module': '06b_hmm_vs_benchmark', 'emissions': list(E.columns),
    'normalization': 'training-only robust z per asset x UTC hour', 'missing_emissions': 'marginalized',
    'k_grid': list(K_VALUES), 'k_selected_by_validation_ll': int(k_sel), 'k_at_grid_edge': bool(k_sel == max(K_VALUES)),
    'train_fraction': TRAIN_FRACTION, 'test_start': str(pd.Timestamp(split.test_start_ns, tz='UTC')),
    'validation_start': str(pd.Timestamp(split.valid_start_ns, tz='UTC')),
    'n_unique_test_onset_timestamps': int(n_unique_test), 'decision': decision,
    'first_time': str(df.state_time.min()), 'last_time': str(df.state_time.max()), 'n_rows': int(len(df)),
}
(OUT / '06b_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote', OUT)
